**1. Import the libraries**

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

**2. Load the data**

In [3]:
file_path = "data/online_retail_II.xlsx"
retail_file = pd.ExcelFile(file_path)
print(retail_file.sheet_names)

['Year 2009-2010', 'Year 2010-2011']


In [6]:
df_2009 = pd.read_excel(retail_file, sheet_name='Year 2009-2010')
df_2010 = pd.read_excel(retail_file, sheet_name='Year 2010-2011')

df = pd.concat([df_2009, df_2010], ignore_index=True)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape: (1067371, 8)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


**3. Check proof of access**

In [7]:
print('Date range:', df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())
print('Unique customers:', df['Customer ID'].nunique())
print('Missing Customer ID:', df['Customer ID'].isna().sum(),
      f"({df['Customer ID'].isna().mean()*100:.1f}%)")

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5942
Missing Customer ID: 243007 (22.8%)


**4. Clean the data**

In [8]:
# Create a copy of data
df_clean = df.copy()
# Drop rows with no Customer ID
df_clean = df.dropna(subset=['Customer ID'])

# Drop rows with non-positive quantity or price (returns, data errors, adjustments)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]

# Drop cancellations — Invoice numbers starting with 'C' are cancelled orders
df_clean['Invoice'] = df_clean['Invoice'].astype(str)
df_clean = df_clean[~df_clean['Invoice'].str.startswith('C')]

# Create a line-item total
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['Price']

# Customer ID as integer (cause it loads as float because of the NaNs we just removed)
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)

print("Original rows:", len(df))
print("Clean rows:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))
print('Retained:', f"{len(df_clean)/len(df)*100:.1f}%")

Original rows: 1067371
Clean rows: 805549
Rows removed: 261822
Retained: 75.5%
